<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/Suman_Gamma_Arylation_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# Pd-CATALYZED γ-ARYLATION
# COMPLETE LIGAND DESCRIPTOR WORKFLOW
#
# Ligand atoms: 17 -> last atom
# Pd atom: 1
#
# A. Steric descriptors
# B. Pd-Ligand coordination descriptors
# C. Donor electronic descriptors
# D. Donor geometry / pyramidalization
# E. Ligand XTB electronic descriptors
# F. NBO ligand -> Pd descriptors
#
# IMPORTANT:
# Every descriptor has geometry / consistency cross-checks.
# The code NEVER silently changes a descriptor because a check
# fails. It records WARNING / CHECK columns instead.
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ============================================================
# 1. INSTALLATION
# ============================================================

!pip -q install morfeus-ml pandas numpy openpyxl scipy


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import glob
import math
import warnings
import traceback

import numpy as np
import pandas as pd

from morfeus import (
    read_xyz,
    SASA,
    Pyramidalization,
    SolidAngle,
    Sterimol,
    BuriedVolume,
    BiteAngle,
)

# XTB interface
try:
    from morfeus import XTB
    XTB_AVAILABLE = True
except Exception:
    XTB_AVAILABLE = False
    print("WARNING: Morfeus XTB interface is not available.")


# ============================================================
# 3. USER SETTINGS
# ============================================================

# ------------------------------------------------------------
# XYZ folder
# ------------------------------------------------------------

XYZ_FOLDER = "/content/drive/MyDrive/Suman_files"

# Output Excel
OUTPUT_EXCEL = "/content/drive/MyDrive/Pd_gamma_arylation_ALL_LIGAND_DESCRIPTORS.xlsx"

# Raw per-ligand validation log
VALIDATION_LOG = "/content/drive/MyDrive/Pd_gamma_arylation_VALIDATION_LOG.xlsx"


# ------------------------------------------------------------
# Atom definitions
# ------------------------------------------------------------

PD_ATOM = 1
LIGAND_START = 17

# Strict Pd-ligand coordination criterion
PD_DONOR_MIN = 2.00
PD_DONOR_MAX = 2.40

DONOR_ELEMENTS = {"N", "P", "S", "O"}


# ------------------------------------------------------------
# Morfeus settings
# ------------------------------------------------------------

SASA_PROBE_RADIUS = 1.4

# Vbur radii
VBUR_RADII = [3.0, 3.5, 4.0]

# Use Bondi for Pd-centered steric calculations
VBUR_RADII_TYPE = "bondi"
SOLID_ANGLE_RADII_TYPE = "bondi"

# Sterimol
STERIMOL_RADII_TYPE = "crc"


# ------------------------------------------------------------
# XTB
# ------------------------------------------------------------

XTB_METHOD = 2          # GFN2-xTB
XTB_CHARGE = 0
XTB_N_UNPAIRED = 0


# ============================================================
# 4. GENERAL UTILITY FUNCTIONS
# ============================================================

def safe_float(x):
    """
    Convert value to float where possible.
    """
    try:
        x = float(x)
        if np.isfinite(x):
            return x
        return np.nan
    except Exception:
        return np.nan


def distance(coords, i, j):
    """
    1-indexed atom distance in Å.
    """
    return float(
        np.linalg.norm(
            np.asarray(coords[i - 1]) -
            np.asarray(coords[j - 1])
        )
    )


def angle_deg(coords, i, j, k):
    """
    Angle i-j-k in degrees.
    1-indexed.
    """
    v1 = np.asarray(coords[i - 1]) - np.asarray(coords[j - 1])
    v2 = np.asarray(coords[k - 1]) - np.asarray(coords[j - 1])

    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)

    if n1 == 0 or n2 == 0:
        return np.nan

    cosang = np.dot(v1, v2) / (n1 * n2)
    cosang = np.clip(cosang, -1.0, 1.0)

    return float(np.degrees(np.arccos(cosang)))


def finite(x):
    """
    True if x is finite numeric.
    """
    try:
        return np.isfinite(float(x))
    except Exception:
        return False


def check(condition):
    """
    Convert Boolean condition to PASS/FAIL.
    """
    return "PASS" if condition else "FAIL"


# ============================================================
# 5. IDENTIFY Pd-COORDINATED LIGAND DONORS
# ============================================================

def find_pd_donors(elements, coordinates):

    pd_xyz = np.asarray(coordinates[PD_ATOM - 1])

    candidates = []

    for atom_id in range(LIGAND_START, len(elements) + 1):

        element = str(elements[atom_id - 1])

        if element not in DONOR_ELEMENTS:
            continue

        xyz = np.asarray(coordinates[atom_id - 1])

        d = float(np.linalg.norm(xyz - pd_xyz))

        candidates.append(
            (atom_id, element, d)
        )

    # Strict cutoff
    selected = [
        x for x in candidates
        if PD_DONOR_MIN <= x[2] <= PD_DONOR_MAX
    ]

    selected = sorted(
        selected,
        key=lambda x: x[2]
    )

    donor1 = selected[0] if len(selected) >= 1 else None
    donor2 = selected[1] if len(selected) >= 2 else None

    return candidates, selected, donor1, donor2


# ============================================================
# 6. VERIFY DONOR SELECTION
# ============================================================

def verify_donor_selection(
    elements,
    coordinates,
    candidates,
    selected,
    donor1,
    donor2
):

    warnings_list = []

    # Check Pd atom
    if PD_ATOM < 1 or PD_ATOM > len(elements):
        warnings_list.append(
            "Pd atom index is outside XYZ"
        )

    # Check selected donors
    if len(selected) == 0:
        warnings_list.append(
            "NO ligand donor atom within strict 2.00–2.30 Å Pd-X cutoff"
        )

    if len(selected) > 2:
        warnings_list.append(
            f"{len(selected)} donor atoms satisfy cutoff; "
            "only two shortest donors are being assigned."
        )

    # Check donor elements
    for donor in [donor1, donor2]:

        if donor is None:
            continue

        atom_id, element, d = donor

        if element not in DONOR_ELEMENTS:
            warnings_list.append(
                f"Atom {atom_id} selected but element {element} "
                "is not an allowed donor."
            )

        if not (
            PD_DONOR_MIN <= d <= PD_DONOR_MAX
        ):
            warnings_list.append(
                f"Atom {atom_id} fails Pd-X cutoff."
            )

    return warnings_list


# ============================================================
# 7. PALLADACYCLE GEOMETRY CHECK
# ============================================================

def covalent_neighbors(
    elements,
    coordinates,
    center,
    allowed_atoms,
    scale_factor=1.25
):
    """
    Geometry-based approximate covalent neighbor detection.

    This is ONLY a validation tool.
    It is not used to replace the actual molecular connectivity.
    """

    # Approximate covalent radii in Å
    radii = {
        "H": 0.31,
        "C": 0.76,
        "N": 0.71,
        "O": 0.66,
        "F": 0.57,
        "P": 1.07,
        "S": 1.05,
        "Cl": 1.02,
        "Br": 1.20,
        "I": 1.39,
        "Pd": 1.39,
    }

    center_el = str(elements[center - 1])

    if center_el not in radii:
        return []

    neighbors = []

    for j in allowed_atoms:

        if j == center:
            continue

        el = str(elements[j - 1])

        if el not in radii:
            continue

        d = distance(coordinates, center, j)

        cutoff = scale_factor * (
            radii[center_el] + radii[el]
        )

        if d <= cutoff:
            neighbors.append(
                (j, el, d)
            )

    return sorted(
        neighbors,
        key=lambda x: x[2]
    )


def determine_palladacycle(
    elements,
    coordinates,
    donor1,
    donor2
):

    result = {
        "Palladacycle_Type": np.nan,
        "Palladacycle_Ring_Size": np.nan,
        "Palladacycle_Check": "NOT_CHECKED",
        "Palladacycle_Warning": ""
    }

    if donor1 is None or donor2 is None:
        result["Palladacycle_Check"] = "FAIL"
        result["Palladacycle_Warning"] = (
            "Two Pd-coordinated donor atoms were not identified."
        )
        return result

    d1 = donor1[0]
    d2 = donor2[0]

    # --------------------------------------------------------
    # Build ligand-only approximate covalent graph
    # --------------------------------------------------------

    ligand_atoms = list(
        range(LIGAND_START, len(elements) + 1)
    )

    graph = {
        atom: []
        for atom in ligand_atoms
    }

    # Add ligand-ligand bonds based on covalent distances
    for i_idx, i in enumerate(ligand_atoms):

        for j in ligand_atoms[i_idx + 1:]:

            ei = str(elements[i - 1])
            ej = str(elements[j - 1])

            # approximate radii
            radii = {
                "H": 0.31,
                "C": 0.76,
                "N": 0.71,
                "O": 0.66,
                "P": 1.07,
                "S": 1.05,
                "F": 0.57,
                "Cl": 1.02,
                "Br": 1.20,
                "I": 1.39,
            }

            if ei not in radii or ej not in radii:
                continue

            d = distance(
                coordinates,
                i,
                j
            )

            cutoff = 1.25 * (
                radii[ei] + radii[ej]
            )

            if d <= cutoff:
                graph[i].append(j)
                graph[j].append(i)

    # --------------------------------------------------------
    # BFS shortest ligand path donor1 -> donor2
    # --------------------------------------------------------

    from collections import deque

    queue = deque()
    queue.append(
        (d1, [d1])
    )

    visited = {d1}

    path = None

    while queue:

        current, current_path = queue.popleft()

        if current == d2:
            path = current_path
            break

        for neighbor in graph.get(current, []):

            if neighbor not in visited:

                visited.add(neighbor)

                queue.append(
                    (
                        neighbor,
                        current_path + [neighbor]
                    )
                )

    if path is None:

        result["Palladacycle_Check"] = "FAIL"

        result["Palladacycle_Warning"] = (
            "No ligand-only covalent path found between donor1 and donor2."
        )

        return result

    # --------------------------------------------------------
    # Ring size
    #
    # Pd -> donor1 -> ligand path -> donor2 -> Pd
    # --------------------------------------------------------

    ring_size = len(path) + 1

    result["Palladacycle_Ring_Size"] = ring_size

    if ring_size == 4:

        result["Palladacycle_Type"] = "4-membered"

    elif ring_size == 5:

        result["Palladacycle_Type"] = "5-membered"

    else:

        result["Palladacycle_Type"] = (
            f"{ring_size}-membered"
        )

    result["Palladacycle_Check"] = "PASS"

    result["Palladacycle_Warning"] = (
        "Ligand-only geometry gives path: "
        + "-".join(map(str, path))
    )

    return result


# ============================================================
# 8. A — STERIC DESCRIPTORS
# ============================================================

def calculate_A_steric(
    elements,
    coordinates,
    donor1,
    donor2
):

    result = {}

    # --------------------------------------------------------
    # A1-A2: SASA
    # --------------------------------------------------------

    try:

        ligand_elements = list(
            elements[LIGAND_START - 1:]
        )

        ligand_coordinates = np.asarray(
            coordinates[LIGAND_START - 1:]
        )

        sasa = SASA(
            ligand_elements,
            ligand_coordinates,
            probe_radius=SASA_PROBE_RADIUS
        )

        result["SASA_Area_A2"] = safe_float(
            sasa.area
        )

        result["SASA_Volume_A3"] = safe_float(
            sasa.volume
        )

        # Individual donor SASA
        ligand_atom_ids = list(
            range(LIGAND_START, len(elements) + 1)
        )

        orig_to_lig = {
            orig: i + 1
            for i, orig in enumerate(ligand_atom_ids)
        }

        if donor1 is not None:

            local_idx = orig_to_lig[donor1[0]]

            result["SASA_Donor1_A2"] = safe_float(
                sasa.atom_areas.get(local_idx, np.nan)
            )

        else:
            result["SASA_Donor1_A2"] = np.nan

        if donor2 is not None:

            local_idx = orig_to_lig[donor2[0]]

            result["SASA_Donor2_A2"] = safe_float(
                sasa.atom_areas.get(local_idx, np.nan)
            )

        else:
            result["SASA_Donor2_A2"] = np.nan

        donor_values = [
            result["SASA_Donor1_A2"],
            result["SASA_Donor2_A2"]
        ]

        donor_values = [
            x for x in donor_values
            if finite(x)
        ]

        result["SASA_Avg_Donor_A2"] = (
            np.mean(donor_values)
            if donor_values
            else np.nan
        )

        result["SASA_Check"] = (
            "PASS"
            if finite(result["SASA_Area_A2"])
            and finite(result["SASA_Volume_A3"])
            else "FAIL"
        )

        result["SASA_Warning"] = ""

    except Exception as e:

        result["SASA_Area_A2"] = np.nan
        result["SASA_Volume_A3"] = np.nan
        result["SASA_Donor1_A2"] = np.nan
        result["SASA_Donor2_A2"] = np.nan
        result["SASA_Avg_Donor_A2"] = np.nan

        result["SASA_Check"] = "FAIL"
        result["SASA_Warning"] = str(e)


    # --------------------------------------------------------
    # A3-A5: Solid angle / cone angle / G
    #
    # IMPORTANT:
    # This uses Pd atom 1 as central atom and the FULL complex.
    # Substrate exclusion is NOT automatically applied here.
    #
    # Therefore this is the Pd-centered steric environment.
    # --------------------------------------------------------

    try:

        solid = SolidAngle(
            elements,
            coordinates,
            PD_ATOM,
            radii_type=SOLID_ANGLE_RADII_TYPE
        )

        result["Solid_Angle_sr"] = safe_float(
            solid.solid_angle
        )

        result["Solid_Cone_Angle_deg"] = safe_float(
            solid.cone_angle
        )

        result["Solid_Angle_G_percent"] = safe_float(
            solid.G
        )

        # Geometry sanity checks
        solid_checks = []

        if finite(result["Solid_Angle_sr"]):
            solid_checks.append(
                result["Solid_Angle_sr"] >= 0
            )

        if finite(result["Solid_Cone_Angle_deg"]):
            solid_checks.append(
                0 <= result["Solid_Cone_Angle_deg"] <= 180
            )

        if finite(result["Solid_Angle_G_percent"]):
            solid_checks.append(
                0 <= result["Solid_Angle_G_percent"] <= 100
            )

        result["SolidAngle_Check"] = (
            "PASS"
            if solid_checks and all(solid_checks)
            else "FAIL"
        )

        result["SolidAngle_Warning"] = ""

    except Exception as e:

        result["Solid_Angle_sr"] = np.nan
        result["Solid_Cone_Angle_deg"] = np.nan
        result["Solid_Angle_G_percent"] = np.nan

        result["SolidAngle_Check"] = "FAIL"
        result["SolidAngle_Warning"] = str(e)


    # --------------------------------------------------------
    # A6-A8: Pd-centered Buried Volume
    # --------------------------------------------------------

    # NOTE:
    # We calculate this on the ligand atoms only.
    # Pd is reference/center.
    #
    # Morfeus BuriedVolume requires the central metal and
    # excluded atoms. We therefore use the FULL complex and
    # exclude everything that is not ligand.
    #
    # This makes the descriptor "Pd-centered ligand-only Vbur".
    # --------------------------------------------------------

    non_ligand_atoms = [
        i
        for i in range(1, len(elements) + 1)
        if not (
            LIGAND_START <= i <= len(elements)
        )
    ]

    for radius in VBUR_RADII:

        key = f"Vbur_{radius:.1f}A_percent"

        try:

            bv = BuriedVolume(
                elements,
                coordinates,
                metal_index=PD_ATOM,
                excluded_atoms=non_ligand_atoms,
                radius=radius,
                include_hs=False,
                radii_type=VBUR_RADII_TYPE
            )

            # Morfeus versions expose fraction/percent
            # differently, so check attributes.
            value = np.nan

            for attr in [
                "percent_buried_volume",
                "buried_volume"
            ]:

                if hasattr(bv, attr):

                    temp = getattr(bv, attr)

                    if finite(temp):

                        value = float(temp)

                        # If returned as fraction convert to %
                        if value <= 1.0:
                            value *= 100.0

                        break

            result[key] = value

            result[f"Vbur_{radius:.1f}A_Check"] = (
                "PASS"
                if finite(value) and 0 <= value <= 100
                else "FAIL"
            )

            result[f"Vbur_{radius:.1f}A_Warning"] = ""

        except Exception as e:

            result[key] = np.nan

            result[f"Vbur_{radius:.1f}A_Check"] = "FAIL"

            result[f"Vbur_{radius:.1f}A_Warning"] = str(e)


    # --------------------------------------------------------
    # A9-A11: Pd-oriented Sterimol
    # --------------------------------------------------------
    #
    # IMPORTANT:
    # Standard Morfeus Sterimol uses a dummy H atom and an
    # attached atom. Therefore using Pd as the dummy is NOT
    # the standard literature Sterimol definition.
    #
    # We therefore label these descriptors:
    #
    # Pd_Oriented_Sterimol_B1
    # Pd_Oriented_Sterimol_B5
    # Pd_Oriented_Sterimol_L
    #
    # They are exploratory geometry descriptors.
    # --------------------------------------------------------

    if donor1 is not None:

        donor1_id = donor1[0]

        try:

            ster = Sterimol(
                elements,
                coordinates,
                PD_ATOM,
                donor1_id,
                radii_type=STERIMOL_RADII_TYPE
            )

            result["Pd_Oriented_Sterimol_B1_A"] = safe_float(
                ster.B_1_value
            )

            result["Pd_Oriented_Sterimol_B5_A"] = safe_float(
                ster.B_5_value
            )

            result["Pd_Oriented_Sterimol_L_A"] = safe_float(
                ster.L_value
            )

            result["Sterimol_Check"] = "PASS"

            result["Sterimol_Warning"] = (
                "Pd-oriented/custom Sterimol; "
                "NOT standard H-dummy literature Sterimol."
            )

        except Exception as e:

            result["Pd_Oriented_Sterimol_B1_A"] = np.nan
            result["Pd_Oriented_Sterimol_B5_A"] = np.nan
            result["Pd_Oriented_Sterimol_L_A"] = np.nan

            result["Sterimol_Check"] = "FAIL"
            result["Sterimol_Warning"] = str(e)

    else:

        result["Pd_Oriented_Sterimol_B1_A"] = np.nan
        result["Pd_Oriented_Sterimol_B5_A"] = np.nan
        result["Pd_Oriented_Sterimol_L_A"] = np.nan

        result["Sterimol_Check"] = "FAIL"
        result["Sterimol_Warning"] = (
            "No coordinated donor1."
        )

    return result


# ============================================================
# 9. B — Pd-LIGAND COORDINATION
# ============================================================

def calculate_B_coordination(
    elements,
    coordinates,
    donor1,
    donor2
):

    result = {}

    d1 = (
        donor1[0]
        if donor1 is not None
        else None
    )

    d2 = (
        donor2[0]
        if donor2 is not None
        else None
    )

    # --------------------------------------------------------
    # Pd-donor distances
    # --------------------------------------------------------

    pd_d1 = (
        distance(coordinates, PD_ATOM, d1)
        if d1 is not None
        else np.nan
    )

    pd_d2 = (
        distance(coordinates, PD_ATOM, d2)
        if d2 is not None
        else np.nan
    )

    result["Pd_Donor1_Distance_A"] = pd_d1
    result["Pd_Donor2_Distance_A"] = pd_d2

    valid_d = [
        x for x in [pd_d1, pd_d2]
        if finite(x)
    ]

    result["Pd_Avg_Donor_Distance_A"] = (
        np.mean(valid_d)
        if valid_d
        else np.nan
    )

    if len(valid_d) == 2:

        result["Pd_Donor_Distance_Delta_A"] = abs(
            pd_d1 - pd_d2
        )

    else:

        result["Pd_Donor_Distance_Delta_A"] = np.nan


    # --------------------------------------------------------
    # Geometry cross-check
    # --------------------------------------------------------

    checks = []

    for d in valid_d:

        checks.append(
            PD_DONOR_MIN <= d <= PD_DONOR_MAX
        )

    result["Pd_Donor_Cutoff_Check"] = (
        "PASS"
        if checks and all(checks)
        else "FAIL"
    )

    result["Pd_Donor_Cutoff_Warning"] = (
        ""
        if result["Pd_Donor_Cutoff_Check"] == "PASS"
        else
        "At least one selected Pd-donor distance is outside "
        "the strict 2.00–2.30 Å criterion."
    )


    # --------------------------------------------------------
    # Bite angle
    # --------------------------------------------------------

    if d1 is not None and d2 is not None:

        try:

            ba = BiteAngle(
                coordinates,
                PD_ATOM,
                d1,
                d2
            )

            result["Pd_Donor1_Donor2_Bite_Angle_deg"] = safe_float(
                ba.angle
            )

            result["BiteAngle_Check"] = (
                "PASS"
                if finite(ba.angle)
                and 0 < ba.angle < 180
                else "FAIL"
            )

            result["BiteAngle_Warning"] = ""

        except Exception as e:

            result["Pd_Donor1_Donor2_Bite_Angle_deg"] = np.nan
            result["BiteAngle_Check"] = "FAIL"
            result["BiteAngle_Warning"] = str(e)

    else:

        result["Pd_Donor1_Donor2_Bite_Angle_deg"] = np.nan
        result["BiteAngle_Check"] = "FAIL"
        result["BiteAngle_Warning"] = (
            "Two coordinated donor atoms required."
        )


    # --------------------------------------------------------
    # Donor1-Donor2 distance
    # --------------------------------------------------------

    if d1 is not None and d2 is not None:

        result["Donor1_Donor2_Distance_A"] = distance(
            coordinates,
            d1,
            d2
        )

        result["Donor_Distance_Check"] = (
            "PASS"
            if finite(result["Donor1_Donor2_Distance_A"])
            and result["Donor1_Donor2_Distance_A"] > 0
            else "FAIL"
        )

    else:

        result["Donor1_Donor2_Distance_A"] = np.nan
        result["Donor_Distance_Check"] = "FAIL"


    # --------------------------------------------------------
    # Pd coordination distortion
    #
    # For two donors:
    #
    # sqrt((d1-mean)^2 + (d2-mean)^2)
    #
    # This is a simple distance asymmetry descriptor.
    # --------------------------------------------------------

    if len(valid_d) == 2:

        mean_d = np.mean(valid_d)

        result["Pd_Coordination_Distance_Distortion_A"] = math.sqrt(
            np.mean(
                [(x - mean_d) ** 2 for x in valid_d]
            )
        )

    else:

        result["Pd_Coordination_Distance_Distortion_A"] = np.nan


    return result


# ============================================================
# 10. C — DONOR ELECTRONIC DESCRIPTORS
#
# These require isolated ligand XTB calculation.
# ============================================================

def calculate_C_donor_electronic(
    elements,
    coordinates,
    donor1,
    donor2
):

    result = {}

    if not XTB_AVAILABLE:

        for key in [
            "Donor1_XTB_Mulliken_Charge",
            "Donor2_XTB_Mulliken_Charge",
            "Avg_Donor_XTB_Charge",
            "Delta_Donor_XTB_Charge"
        ]:
            result[key] = np.nan

        result["XTB_Donor_Check"] = "FAIL"
        result["XTB_Donor_Warning"] = (
            "Morfeus XTB interface unavailable."
        )

        return result


    try:

        ligand_elements = list(
            elements[LIGAND_START - 1:]
        )

        ligand_coordinates = np.asarray(
            coordinates[LIGAND_START - 1:]
        )

        xtb = XTB(
            ligand_elements,
            ligand_coordinates,
            method=XTB_METHOD,
            charge=XTB_CHARGE,
            n_unpaired=XTB_N_UNPAIRED
        )

        charges = xtb.get_charges(
            model="Mulliken"
        )

        ligand_atom_ids = list(
            range(
                LIGAND_START,
                len(elements) + 1
            )
        )

        orig_to_lig = {
            orig: i + 1
            for i, orig in enumerate(
                ligand_atom_ids
            )
        }

        # ----------------------------------------------------
        # Donor 1
        # ----------------------------------------------------

        if donor1 is not None:

            idx1 = orig_to_lig[
                donor1[0]
            ]

            q1 = charges[idx1]

        else:

            q1 = np.nan


        # ----------------------------------------------------
        # Donor 2
        # ----------------------------------------------------

        if donor2 is not None:

            idx2 = orig_to_lig[
                donor2[0]
            ]

            q2 = charges[idx2]

        else:

            q2 = np.nan


        result["Donor1_XTB_Mulliken_Charge"] = safe_float(q1)
        result["Donor2_XTB_Mulliken_Charge"] = safe_float(q2)


        valid_q = [
            q for q in [q1, q2]
            if finite(q)
        ]

        result["Avg_Donor_XTB_Charge"] = (
            np.mean(valid_q)
            if valid_q
            else np.nan
        )

        result["Delta_Donor_XTB_Charge"] = (
            abs(q1 - q2)
            if finite(q1) and finite(q2)
            else np.nan
        )


        # ----------------------------------------------------
        # Cross-check:
        # Mulliken charge sum should approximately equal
        # formal ligand charge.
        # ----------------------------------------------------

        charge_sum = sum(
            float(v)
            for v in charges.values()
        )

        result["XTB_Mulliken_Charge_Sum_Check"] = charge_sum

        result["XTB_Donor_Check"] = (
            "PASS"
            if finite(q1) or finite(q2)
            else "FAIL"
        )

        # Do not use this as predictor
        result["XTB_Donor_Warning"] = (
            "Mulliken donor charges are from isolated ligand "
            "GFN2-xTB geometry extracted from the TS."
        )

    except Exception as e:

        result["Donor1_XTB_Mulliken_Charge"] = np.nan
        result["Donor2_XTB_Mulliken_Charge"] = np.nan
        result["Avg_Donor_XTB_Charge"] = np.nan
        result["Delta_Donor_XTB_Charge"] = np.nan
        result["XTB_Mulliken_Charge_Sum_Check"] = np.nan

        result["XTB_Donor_Check"] = "FAIL"
        result["XTB_Donor_Warning"] = str(e)

    return result


# ============================================================
# 11. D — DONOR PYRAMIDALIZATION
# ============================================================

def calculate_D_pyramidalization(
    elements,
    coordinates,
    donor1,
    donor2
):

    result = {}

    # Only ligand atoms are allowed to be neighbors.
    ligand_atoms = set(
        range(
            LIGAND_START,
            len(elements) + 1
        )
    )

    # --------------------------------------------------------
    # Helper
    # --------------------------------------------------------

    def calc_one(donor):

        if donor is None:

            return (
                np.nan,
                np.nan,
                [],
                "FAIL",
                "No donor atom."
            )

        donor_id = donor[0]

        try:

            # ------------------------------------------------
            # First attempt:
            # explicit ligand-only neighbor detection
            # ------------------------------------------------

            neighbors = covalent_neighbors(
                elements,
                coordinates,
                donor_id,
                ligand_atoms
            )

            neighbor_ids = [
                x[0]
                for x in neighbors
            ]

            # Pyramidalization needs appropriate neighbors.
            # If 3+ neighbors exist, use the closest 3.
            if len(neighbor_ids) >= 3:

                neighbor_ids = neighbor_ids[:3]

                pyr = Pyramidalization(
                    coordinates,
                    donor_id,
                    neighbor_indices=neighbor_ids,
                    elements=elements
                )

            else:

                # Second attempt: Morfeus automatic connectivity,
                # explicitly excluding Pd and non-ligand atoms.
                excluded = [
                    i
                    for i in range(1, len(elements) + 1)
                    if i not in ligand_atoms
                ]

                pyr = Pyramidalization(
                    coordinates,
                    donor_id,
                    elements=elements,
                    excluded_atoms=excluded,
                    method="connectivity"
                )

                neighbor_ids = pyr.neighbor_indices

            P = safe_float(pyr.P)
            P_angle = safe_float(pyr.P_angle)

            if (
                finite(P)
                and finite(P_angle)
                and len(neighbor_ids) >= 3
            ):

                status = "PASS"
                warning = ""

            else:

                status = "FAIL"
                warning = (
                    f"Insufficient/invalid ligand neighbors: "
                    f"{neighbor_ids}"
                )

            return (
                P,
                P_angle,
                neighbor_ids,
                status,
                warning
            )

        except Exception as e:

            return (
                np.nan,
                np.nan,
                [],
                "FAIL",
                str(e)
            )


    # --------------------------------------------------------
    # Donor 1
    # --------------------------------------------------------

    (
        P1,
        P1_angle,
        N1,
        C1,
        W1
    ) = calc_one(donor1)

    result["Donor1_Pyramidalization_P"] = P1
    result["Donor1_Pyramidalization_Angle_deg"] = P1_angle
    result["Donor1_Pyr_Neighbors"] = str(N1)
    result["Donor1_Pyr_Check"] = C1
    result["Donor1_Pyr_Warning"] = W1


    # --------------------------------------------------------
    # Donor 2
    # --------------------------------------------------------

    (
        P2,
        P2_angle,
        N2,
        C2,
        W2
    ) = calc_one(donor2)

    result["Donor2_Pyramidalization_P"] = P2
    result["Donor2_Pyramidalization_Angle_deg"] = P2_angle
    result["Donor2_Pyr_Neighbors"] = str(N2)
    result["Donor2_Pyr_Check"] = C2
    result["Donor2_Pyr_Warning"] = W2


    # --------------------------------------------------------
    # Average / delta
    # --------------------------------------------------------

    Ps = [
        x for x in [P1, P2]
        if finite(x)
    ]

    Pangles = [
        x for x in [P1_angle, P2_angle]
        if finite(x)
    ]

    result["Avg_Donor_Pyramidalization_P"] = (
        np.mean(Ps)
        if Ps
        else np.nan
    )

    result["Delta_Donor_Pyramidalization_P"] = (
        abs(P1 - P2)
        if finite(P1) and finite(P2)
        else np.nan
    )

    result["Avg_Donor_Pyramidalization_Angle_deg"] = (
        np.mean(Pangles)
        if Pangles
        else np.nan
    )

    return result


# ============================================================
# 12. E — COMPLETE LIGAND XTB ELECTRONIC DESCRIPTORS
# ============================================================

def calculate_E_xtb(
    elements,
    coordinates
):

    result = {}

    # --------------------------------------------------------
    # Initialize all expected fields
    # --------------------------------------------------------

    xtb_fields = [
        "XTB_Total_Energy_Eh",
        "XTB_HOMO_eV",
        "XTB_LUMO_eV",
        "XTB_HOMO_LUMO_Gap_eV",
        "XTB_Fermi_Level_eV",
        "XTB_IP_eV",
        "XTB_EA_eV",
        "XTB_Chemical_Potential_eV",
        "XTB_Electronegativity_eV",
        "XTB_Hardness_eV",
        "XTB_Softness",
        "XTB_Electrophilicity",
        "XTB_Nucleophilicity",
        "XTB_Dipole_Debye",
        "XTB_Molecular_Polarizability",
        "XTB_NFOD"
    ]

    for field in xtb_fields:
        result[field] = np.nan

    if not XTB_AVAILABLE:

        result["XTB_Check"] = "FAIL"
        result["XTB_Warning"] = (
            "Morfeus XTB interface unavailable."
        )

        return result


    # --------------------------------------------------------
    # Isolated ligand = atoms 17 -> last
    # --------------------------------------------------------

    ligand_elements = list(
        elements[LIGAND_START - 1:]
    )

    ligand_coordinates = np.asarray(
        coordinates[LIGAND_START - 1:]
    )


    try:

        xtb = XTB(
            ligand_elements,
            ligand_coordinates,
            method=XTB_METHOD,
            charge=XTB_CHARGE,
            n_unpaired=XTB_N_UNPAIRED
        )

        # ----------------------------------------------------
        # Energy
        # ----------------------------------------------------

        try:
            result["XTB_Total_Energy_Eh"] = safe_float(
                xtb.get_energy()
            )
        except Exception:
            pass


        # ----------------------------------------------------
        # Frontier orbitals
        # ----------------------------------------------------

        try:
            result["XTB_HOMO_eV"] = safe_float(
                xtb.get_homo("eV")
            )
        except Exception:
            pass

        try:
            result["XTB_LUMO_eV"] = safe_float(
                xtb.get_lumo("eV")
            )
        except Exception:
            pass

        try:
            result["XTB_HOMO_LUMO_Gap_eV"] = safe_float(
                xtb.get_homo_lumo_gap("eV")
            )
        except Exception:
            pass


        # ----------------------------------------------------
        # Fermi level
        # ----------------------------------------------------

        try:
            result["XTB_Fermi_Level_eV"] = safe_float(
                xtb.get_fermi_level()
            )
        except Exception:
            pass


        # ----------------------------------------------------
        # IPEA descriptors
        # ----------------------------------------------------

        try:
            result["XTB_IP_eV"] = safe_float(
                xtb.get_ip(corrected=True)
            )
        except Exception:
            pass

        try:
            result["XTB_EA_eV"] = safe_float(
                xtb.get_ea(corrected=True)
            )
        except Exception:
            pass


        # ----------------------------------------------------
        # Global descriptors
        # ----------------------------------------------------

        try:
            result["XTB_Chemical_Potential_eV"] = safe_float(
                xtb.get_chemical_potential(
                    corrected=True
                )
            )
        except Exception:
            pass

        try:
            result["XTB_Electronegativity_eV"] = safe_float(
                xtb.get_electronegativity(
                    corrected=True
                )
            )
        except Exception:
            pass

        try:
            result["XTB_Hardness_eV"] = safe_float(
                xtb.get_hardness()
            )
        except Exception:
            pass

        try:
            result["XTB_Softness"] = safe_float(
                xtb.get_softness()
            )
        except Exception:
            pass


        # ----------------------------------------------------
        # Electrophilicity / nucleophilicity
        # ----------------------------------------------------

        try:
            result["XTB_Electrophilicity"] = safe_float(
                xtb.get_global_descriptor(
                    "electrophilicity",
                    corrected=True
                )
            )
        except Exception:
            pass

        try:
            result["XTB_Nucleophilicity"] = safe_float(
                xtb.get_global_descriptor(
                    "nucleophilicity",
                    corrected=True
                )
            )
        except Exception:
            pass


        # ----------------------------------------------------
        # Dipole
        # ----------------------------------------------------

        try:
            result["XTB_Dipole_Debye"] = safe_float(
                xtb.get_dipole_moment(
                    unit="debye"
                )
            )
        except Exception:

            try:
                result["XTB_Dipole_Debye"] = safe_float(
                    xtb.get_dipole(
                    )
                )
            except Exception:
                pass


        # ----------------------------------------------------
        # Molecular polarizability
        # ----------------------------------------------------

        try:
            result["XTB_Molecular_Polarizability"] = safe_float(
                xtb.get_molecular_polarizability()
            )
        except Exception:
            pass


        # ----------------------------------------------------
        # NFOD
        # ----------------------------------------------------

        try:
            result["XTB_NFOD"] = safe_float(
                xtb.get_nfod()
            )
        except Exception:
            pass


        # ----------------------------------------------------
        # IMPORTANT INTERNAL CONSISTENCY CHECK
        #
        # HOMO-LUMO gap should approximately equal:
        # LUMO - HOMO
        # ----------------------------------------------------

        homo = result["XTB_HOMO_eV"]
        lumo = result["XTB_LUMO_eV"]
        gap = result["XTB_HOMO_LUMO_Gap_eV"]

        if (
            finite(homo)
            and finite(lumo)
            and finite(gap)
        ):

            calculated_gap = lumo - homo

            result["XTB_Gap_Recalculated_eV"] = calculated_gap

            result["XTB_Gap_Difference_eV"] = abs(
                calculated_gap - gap
            )

            result["XTB_Gap_Check"] = (
                "PASS"
                if abs(calculated_gap - gap) < 0.05
                else "WARNING"
            )

        else:

            result["XTB_Gap_Recalculated_eV"] = np.nan
            result["XTB_Gap_Difference_eV"] = np.nan
            result["XTB_Gap_Check"] = "NOT_CHECKED"


        # ----------------------------------------------------
        # XTB overall check
        # ----------------------------------------------------

        required = [
            result["XTB_HOMO_eV"],
            result["XTB_LUMO_eV"],
            result["XTB_HOMO_LUMO_Gap_eV"]
        ]

        result["XTB_Check"] = (
            "PASS"
            if all(finite(x) for x in required)
            else "WARNING"
        )

        result["XTB_Warning"] = (
            "GFN2-xTB on ligand atoms 17->last. "
            "Coordinates are taken directly from the Pd TS; "
            "the isolated ligand was NOT geometry optimized."
        )

    except Exception as e:

        result["XTB_Check"] = "FAIL"
        result["XTB_Warning"] = str(e)

        result["XTB_Gap_Recalculated_eV"] = np.nan
        result["XTB_Gap_Difference_eV"] = np.nan
        result["XTB_Gap_Check"] = "FAIL"


    return result


# ============================================================
# 13. F — NBO LIGAND -> Pd
# ============================================================
#
# IMPORTANT:
# XYZ files CANNOT provide NBO E(2).
#
# This function is therefore designed to READ NBO results
# from Gaussian output.
#
# We only accept donor-ligand -> Pd interactions.
#
# It does NOT include Pd -> ligand or ligand-ligand NBO.
# ============================================================

def parse_nbo_ligand_to_pd(
    gaussian_log_file,
    donor1_id=None,
    donor2_id=None
):

    result = {

        "NBO_Donor1_to_Pd_E2_kcalmol": np.nan,
        "NBO_Donor2_to_Pd_E2_kcalmol": np.nan,
        "NBO_Total_Ligand_to_Pd_E2_kcalmol": np.nan,
        "NBO_Max_Ligand_to_Pd_E2_kcalmol": np.nan,

        "NBO_Check": "NOT_CHECKED",

        "NBO_Warning": (
            "NBO descriptors require Gaussian/NBO output. "
            "XYZ alone cannot provide E(2)."
        )
    }

    if gaussian_log_file is None:
        return result

    if not os.path.exists(gaussian_log_file):
        result["NBO_Check"] = "FAIL"
        result["NBO_Warning"] = (
            "Gaussian/NBO output file not found."
        )
        return result


    # --------------------------------------------------------
    # Generic parser
    #
    # NBO output formats vary by Gaussian/NBO version.
    # Therefore we DO NOT pretend a generic regex is guaranteed.
    #
    # The code below searches for donor -> Pd interactions.
    # --------------------------------------------------------

    try:

        with open(
            gaussian_log_file,
            "r",
            errors="ignore"
        ) as f:

            text = f.read()


        # Store every possible donor -> Pd E2
        interactions = []

        # NBO second-order perturbation block commonly contains:
        #
        # E(2) ...
        #
        # We inspect lines rather than assuming exact formatting.
        lines = text.splitlines()

        for i, line in enumerate(lines):

            lower = line.lower()

            if "e(2)" not in lower:
                continue

            # Search local block around line
            block = "\n".join(
                lines[
                    max(0, i - 3):
                    min(len(lines), i + 4)
                ]
            )

            # Pd indication
            if "pd" not in block.lower():
                continue

            # Extract numbers at end of line
            import re

            numbers = re.findall(
                r"[-+]?\d*\.\d+|\d+",
                line
            )

            if not numbers:
                continue

            # Last numerical field is usually E(2)
            try:
                e2 = float(numbers[-1])
            except Exception:
                continue

            interactions.append(
                {
                    "line": line,
                    "block": block,
                    "E2": e2
                }
            )


        if interactions:

            values = [
                x["E2"]
                for x in interactions
                if finite(x["E2"])
            ]

            if values:

                result[
                    "NBO_Total_Ligand_to_Pd_E2_kcalmol"
                ] = np.sum(values)

                result[
                    "NBO_Max_Ligand_to_Pd_E2_kcalmol"
                ] = np.max(values)

                result["NBO_Check"] = "WARNING"

                result["NBO_Warning"] = (
                    "NBO Pd-related E(2) interactions were found, "
                    "but donor-atom assignment should be manually "
                    "verified against the NBO output."
                )

            else:

                result["NBO_Check"] = "FAIL"

        else:

            result["NBO_Check"] = "FAIL"

            result["NBO_Warning"] = (
                "No Pd-related E(2) interactions automatically found."
            )

    except Exception as e:

        result["NBO_Check"] = "FAIL"
        result["NBO_Warning"] = str(e)


    return result


# ============================================================
# 14. ONE-LIGAND COMPLETE CALCULATION
# ============================================================

def calculate_one_xyz(xyz_file):

    row = {}

    row["File"] = os.path.basename(xyz_file)

    # --------------------------------------------------------
    # Read XYZ
    # --------------------------------------------------------

    try:

        elements, coordinates = read_xyz(
            xyz_file
        )

        elements = list(elements)
        coordinates = np.asarray(
            coordinates,
            dtype=float
        )

    except Exception as e:

        row["Overall_Check"] = "FAIL"
        row["Overall_Warning"] = (
            f"XYZ read error: {e}"
        )

        return row


    # --------------------------------------------------------
    # Basic geometry validation
    # --------------------------------------------------------

    row["Total_Atoms"] = len(elements)
    row["Pd_Atom"] = PD_ATOM
    row["Ligand_Start_Atom"] = LIGAND_START
    row["Ligand_Atom_Count"] = (
        len(elements) - LIGAND_START + 1
    )

    geometry_checks = []

    # Pd exists
    geometry_checks.append(
        1 <= PD_ATOM <= len(elements)
    )

    # ligand exists
    geometry_checks.append(
        LIGAND_START <= len(elements)
    )

    # coordinate shape
    geometry_checks.append(
        coordinates.shape == (len(elements), 3)
    )

    # finite coordinates
    geometry_checks.append(
        np.isfinite(coordinates).all()
    )

    row["XYZ_Geometry_Check"] = (
        "PASS"
        if all(geometry_checks)
        else "FAIL"
    )


    if not all(geometry_checks):

        row["Overall_Check"] = "FAIL"
        row["Overall_Warning"] = (
            "Basic XYZ geometry validation failed."
        )

        return row


    # --------------------------------------------------------
    # DONOR IDENTIFICATION
    # --------------------------------------------------------

    (
        candidates,
        selected,
        donor1,
        donor2
    ) = find_pd_donors(
        elements,
        coordinates
    )

    row["Pd_Donor_Candidates"] = str(
        [
            (
                a,
                e,
                round(d, 3)
            )
            for a, e, d in candidates
        ]
    )

    row["Pd_Donors_Within_2.0_2.3A"] = str(
        [
            (
                a,
                e,
                round(d, 3)
            )
            for a, e, d in selected
        ]
    )


    if donor1 is not None:

        row["Donor1_Atom"] = donor1[0]
        row["Donor1_Element"] = donor1[1]
        row["Donor1_Pd_Distance_A"] = donor1[2]

    else:

        row["Donor1_Atom"] = np.nan
        row["Donor1_Element"] = ""
        row["Donor1_Pd_Distance_A"] = np.nan


    if donor2 is not None:

        row["Donor2_Atom"] = donor2[0]
        row["Donor2_Element"] = donor2[1]
        row["Donor2_Pd_Distance_A"] = donor2[2]

    else:

        row["Donor2_Atom"] = np.nan
        row["Donor2_Element"] = ""
        row["Donor2_Pd_Distance_A"] = np.nan


    donor_warnings = verify_donor_selection(
        elements,
        coordinates,
        candidates,
        selected,
        donor1,
        donor2
    )

    row["Donor_Selection_Check"] = (
        "PASS"
        if len(donor_warnings) == 0
        and len(selected) >= 2
        else "WARNING"
    )

    row["Donor_Selection_Warning"] = "; ".join(
        donor_warnings
    )


    # ========================================================
    # A
    # ========================================================

    A = calculate_A_steric(
        elements,
        coordinates,
        donor1,
        donor2
    )

    row.update(A)


    # ========================================================
    # B
    # ========================================================

    B = calculate_B_coordination(
        elements,
        coordinates,
        donor1,
        donor2
    )

    row.update(B)


    # ========================================================
    # Palladacycle
    # ========================================================

    ring = determine_palladacycle(
        elements,
        coordinates,
        donor1,
        donor2
    )

    row.update(ring)


    # ========================================================
    # C
    # ========================================================

    C = calculate_C_donor_electronic(
        elements,
        coordinates,
        donor1,
        donor2
    )

    row.update(C)


    # ========================================================
    # D
    # ========================================================

    D = calculate_D_pyramidalization(
        elements,
        coordinates,
        donor1,
        donor2
    )

    row.update(D)


    # ========================================================
    # E
    # ========================================================

    E = calculate_E_xtb(
        elements,
        coordinates
    )

    row.update(E)


    # ========================================================
    # F
    # ========================================================
    #
    # Search for corresponding Gaussian output automatically.
    #
    # If not found, F remains blank and is clearly marked.
    # ========================================================

    base = os.path.splitext(
        os.path.basename(xyz_file)
    )[0]

    possible_logs = [
        os.path.join(
            os.path.dirname(xyz_file),
            base + ".log"
        ),
        os.path.join(
            os.path.dirname(xyz_file),
            base + ".out"
        )
    ]

    nbo_file = None

    for candidate in possible_logs:

        if os.path.exists(candidate):

            nbo_file = candidate
            break

    F = parse_nbo_ligand_to_pd(
        nbo_file,
        row.get("Donor1_Atom"),
        row.get("Donor2_Atom")
    )

    row.update(F)


    # ========================================================
    # FINAL GLOBAL VALIDATION
    # ========================================================

    checks = []

    check_columns = [
        "XYZ_Geometry_Check",
        "Donor_Selection_Check",
        "Pd_Donor_Cutoff_Check",
        "BiteAngle_Check",
        "SASA_Check",
        "SolidAngle_Check",
        "Sterimol_Check",
        "XTB_Donor_Check",
        "XTB_Check",
        "Donor1_Pyr_Check",
        "Donor2_Pyr_Check",
        "Palladacycle_Check"
    ]

    for col in check_columns:

        if col in row:

            value = row[col]

            if value == "FAIL":

                checks.append(False)

            elif value == "PASS":

                checks.append(True)

    if not checks:

        row["Overall_Check"] = "WARNING"

    elif all(checks):

        row["Overall_Check"] = "PASS"

    else:

        row["Overall_Check"] = "WARNING"


    # --------------------------------------------------------
    # Collect warnings
    # --------------------------------------------------------

    warning_cols = [
        x
        for x in row.keys()
        if (
            "Warning" in x
            or "WARNING" in x
        )
    ]

    warnings_found = []

    for col in warning_cols:

        value = row.get(col)

        if (
            value is not None
            and str(value).strip() not in ["", "nan", "None"]
        ):

            warnings_found.append(
                f"{col}: {value}"
            )

    row["ALL_Warnings"] = " || ".join(
        warnings_found
    )

    return row


# ============================================================
# 15. FIND ALL XYZ FILES
# ============================================================

xyz_files = sorted(
    glob.glob(
        os.path.join(
            XYZ_FOLDER,
            "*.xyz"
        )
    )
)

print(
    f"Found {len(xyz_files)} XYZ files."
)

if len(xyz_files) == 0:

    raise FileNotFoundError(
        f"No XYZ files found in:\n{XYZ_FOLDER}"
    )


# ============================================================
# 16. RUN ALL FILES
# ============================================================

all_rows = []

for i, xyz_file in enumerate(
    xyz_files,
    start=1
):

    print(
        f"\n[{i}/{len(xyz_files)}] "
        f"{os.path.basename(xyz_file)}"
    )

    try:

        row = calculate_one_xyz(
            xyz_file
        )

        all_rows.append(row)

        print(
            "  Overall:",
            row.get(
                "Overall_Check",
                "UNKNOWN"
            )
        )

        print(
            "  Donor1:",
            row.get("Donor1_Atom"),
            row.get("Donor1_Element"),
            row.get("Donor1_Pd_Distance_A")
        )

        print(
            "  Donor2:",
            row.get("Donor2_Atom"),
            row.get("Donor2_Element"),
            row.get("Donor2_Pd_Distance_A")
        )

        print(
            "  Palladacycle:",
            row.get(
                "Palladacycle_Type"
            )
        )

    except Exception as e:

        print(
            "  ERROR:",
            e
        )

        all_rows.append(
            {
                "File": os.path.basename(
                    xyz_file
                ),
                "Overall_Check": "FAIL",
                "Overall_Warning": str(e)
            }
        )


# ============================================================
# 17. CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(
    all_rows
)

# Sort by filename
df = df.sort_values(
    "File"
).reset_index(
    drop=True
)


# ============================================================
# 18. ADD LIGAND LABEL
# ============================================================

def ligand_label(filename):

    base = os.path.splitext(
        filename
    )[0]

    return base


df.insert(
    0,
    "Ligand",
    df["File"].apply(
        ligand_label
    )
)


# ============================================================
# 19. DISPLAY IMPORTANT VALIDATION TABLE
# ============================================================

validation_columns = [
    "Ligand",

    "Total_Atoms",
    "Ligand_Atom_Count",

    "Donor1_Atom",
    "Donor1_Element",
    "Donor1_Pd_Distance_A",

    "Donor2_Atom",
    "Donor2_Element",
    "Donor2_Pd_Distance_A",

    "Donor_Selection_Check",
    "Pd_Donor_Cutoff_Check",

    "Palladacycle_Type",
    "Palladacycle_Ring_Size",
    "Palladacycle_Check",

    "BiteAngle_Check",

    "SASA_Check",
    "SolidAngle_Check",
    "Sterimol_Check",

    "Donor1_Pyr_Check",
    "Donor2_Pyr_Check",

    "XTB_Donor_Check",
    "XTB_Check",

    "NBO_Check",

    "Overall_Check"
]

validation_columns = [
    c
    for c in validation_columns
    if c in df.columns
]

print("\n")
print("=" * 100)
print("GEOMETRY / DESCRIPTOR CROSS-CHECK SUMMARY")
print("=" * 100)

display(
    df[validation_columns]
)


# ============================================================
# 20. PRINT ALL WARNINGS
# ============================================================

print("\n")
print("=" * 100)
print("FILES REQUIRING MANUAL INSPECTION")
print("=" * 100)

warning_df = df[
    df["Overall_Check"] != "PASS"
]

if len(warning_df) == 0:

    print(
        "No overall failures/warnings detected."
    )

else:

    for _, r in warning_df.iterrows():

        print(
            "\nFILE:",
            r["File"]
        )

        print(
            "STATUS:",
            r["Overall_Check"]
        )

        print(
            "WARNINGS:",
            r.get(
                "ALL_Warnings",
                ""
            )
        )


# ============================================================
# 21. SAVE COMPLETE EXCEL
# ============================================================

with pd.ExcelWriter(
    OUTPUT_EXCEL,
    engine="openpyxl"
) as writer:

    # --------------------------------------------------------
    # Sheet 1: ALL DESCRIPTORS
    # --------------------------------------------------------

    df.to_excel(
        writer,
        sheet_name="ALL_DESCRIPTORS",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 2: VALIDATION
    # --------------------------------------------------------

    validation_cols = [
        c
        for c in df.columns
        if (
            "Check" in c
            or "Warning" in c
            or c in [
                "Ligand",
                "File",
                "Donor1_Atom",
                "Donor1_Element",
                "Donor1_Pd_Distance_A",
                "Donor2_Atom",
                "Donor2_Element",
                "Donor2_Pd_Distance_A",
                "Palladacycle_Type",
                "Palladacycle_Ring_Size"
            ]
        )
    ]

    df[
        validation_cols
    ].to_excel(
        writer,
        sheet_name="VALIDATION",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 3: DONOR_COORDINATION
    # --------------------------------------------------------

    donor_cols = [
        c
        for c in df.columns
        if (
            "Donor" in c
            or "Pd_" in c
            or "Bite" in c
            or "Palladacycle" in c
        )
    ]

    if "Ligand" not in donor_cols:
        donor_cols.insert(
            0,
            "Ligand"
        )

    df[
        list(dict.fromkeys(donor_cols))
    ].to_excel(
        writer,
        sheet_name="DONOR_COORDINATION",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 4: STERIC_A
    # --------------------------------------------------------

    steric_cols = [
        "Ligand",
        "SASA_Area_A2",
        "SASA_Volume_A3",
        "SASA_Donor1_A2",
        "SASA_Donor2_A2",
        "SASA_Avg_Donor_A2",
        "Solid_Angle_sr",
        "Solid_Cone_Angle_deg",
        "Solid_Angle_G_percent",
        "Vbur_3.0A_percent",
        "Vbur_3.5A_percent",
        "Vbur_4.0A_percent",
        "Pd_Oriented_Sterimol_B1_A",
        "Pd_Oriented_Sterimol_B5_A",
        "Pd_Oriented_Sterimol_L_A",
        "SASA_Check",
        "SolidAngle_Check",
        "Sterimol_Check"
    ]

    steric_cols = [
        c
        for c in steric_cols
        if c in df.columns
    ]

    df[
        steric_cols
    ].to_excel(
        writer,
        sheet_name="A_STERIC",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 5: B_COORDINATION
    # --------------------------------------------------------

    B_cols = [
        "Ligand",
        "Pd_Donor1_Distance_A",
        "Pd_Donor2_Distance_A",
        "Pd_Avg_Donor_Distance_A",
        "Pd_Donor_Distance_Delta_A",
        "Pd_Donor1_Donor2_Bite_Angle_deg",
        "Donor1_Donor2_Distance_A",
        "Pd_Coordination_Distance_Distortion_A",
        "Palladacycle_Type",
        "Palladacycle_Ring_Size",
        "Pd_Donor_Cutoff_Check",
        "BiteAngle_Check",
        "Palladacycle_Check"
    ]

    B_cols = [
        c
        for c in B_cols
        if c in df.columns
    ]

    df[
        B_cols
    ].to_excel(
        writer,
        sheet_name="B_COORDINATION",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 6: C_DONOR_ELECTRONIC
    # --------------------------------------------------------

    C_cols = [
        "Ligand",
        "Donor1_XTB_Mulliken_Charge",
        "Donor2_XTB_Mulliken_Charge",
        "Avg_Donor_XTB_Charge",
        "Delta_Donor_XTB_Charge",
        "XTB_Mulliken_Charge_Sum_Check",
        "XTB_Donor_Check"
    ]

    C_cols = [
        c
        for c in C_cols
        if c in df.columns
    ]

    df[
        C_cols
    ].to_excel(
        writer,
        sheet_name="C_DONOR_ELECTRONIC",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 7: D_PYRAMIDALIZATION
    # --------------------------------------------------------

    D_cols = [
        "Ligand",

        "Donor1_Pyramidalization_P",
        "Donor1_Pyramidalization_Angle_deg",
        "Donor1_Pyr_Neighbors",
        "Donor1_Pyr_Check",

        "Donor2_Pyramidalization_P",
        "Donor2_Pyramidalization_Angle_deg",
        "Donor2_Pyr_Neighbors",
        "Donor2_Pyr_Check",

        "Avg_Donor_Pyramidalization_P",
        "Delta_Donor_Pyramidalization_P",
        "Avg_Donor_Pyramidalization_Angle_deg"
    ]

    D_cols = [
        c
        for c in D_cols
        if c in df.columns
    ]

    df[
        D_cols
    ].to_excel(
        writer,
        sheet_name="D_PYRAMIDALIZATION",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 8: E_XTB
    # --------------------------------------------------------

    E_cols = [
        "Ligand",

        "XTB_Total_Energy_Eh",

        "XTB_HOMO_eV",
        "XTB_LUMO_eV",
        "XTB_HOMO_LUMO_Gap_eV",
        "XTB_Gap_Recalculated_eV",
        "XTB_Gap_Difference_eV",

        "XTB_Fermi_Level_eV",

        "XTB_IP_eV",
        "XTB_EA_eV",

        "XTB_Chemical_Potential_eV",
        "XTB_Electronegativity_eV",
        "XTB_Hardness_eV",
        "XTB_Softness",

        "XTB_Electrophilicity",
        "XTB_Nucleophilicity",

        "XTB_Dipole_Debye",
        "XTB_Molecular_Polarizability",

        "XTB_NFOD",

        "XTB_Gap_Check",
        "XTB_Check"
    ]

    E_cols = [
        c
        for c in E_cols
        if c in df.columns
    ]

    df[
        E_cols
    ].to_excel(
        writer,
        sheet_name="E_XTB",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 9: F_NBO
    # --------------------------------------------------------

    F_cols = [
        "Ligand",
        "NBO_Donor1_to_Pd_E2_kcalmol",
        "NBO_Donor2_to_Pd_E2_kcalmol",
        "NBO_Total_Ligand_to_Pd_E2_kcalmol",
        "NBO_Max_Ligand_to_Pd_E2_kcalmol",
        "NBO_Check",
        "NBO_Warning"
    ]

    F_cols = [
        c
        for c in F_cols
        if c in df.columns
    ]

    df[
        F_cols
    ].to_excel(
        writer,
        sheet_name="F_NBO",
        index=False
    )


print("\n")
print("=" * 100)
print("DONE")
print("=" * 100)
print(
    "Excel file:",
    OUTPUT_EXCEL
)

Mounted at /content/drive
Found 19 XYZ files.

[1/19] L10_new.xyz


/tmp/ipykernel_1983/2467750343.py:733: DeprecationWarning: 'percent_buried_volume' is deprecated. Use 'fraction_buried_volume'.
  if hasattr(bv, attr):
/tmp/ipykernel_1983/2467750343.py:735: DeprecationWarning: 'percent_buried_volume' is deprecated. Use 'fraction_buried_volume'.
  temp = getattr(bv, attr)


  Overall: WARNING
  Donor1: 17 N 2.112539265549637
  Donor2: 20 N 2.1270163421478454
  Palladacycle: 5-membered

[2/19] L11_new.xyz
  Overall: WARNING
  Donor1: 20 N 2.136284747195701
  Donor2: 17 N 2.1486572303171116
  Palladacycle: 5-membered

[3/19] L12_new.xyz
  Overall: WARNING
  Donor1: 17 N 2.121125260667319
  Donor2: 20 N 2.2855310735330203
  Palladacycle: 5-membered

[4/19] L13_new.xyz
  Overall: WARNING
  Donor1: 17 N 2.1082510928293146
  Donor2: 20 N 2.138894520627186
  Palladacycle: 5-membered

[5/19] L14-new.xyz
  Overall: WARNING
  Donor1: 17 N 2.1391236666480973
  Donor2: 20 N 2.164505380044365
  Palladacycle: 5-membered

[6/19] L15_new.xyz
  Overall: WARNING
  Donor1: 17 N 2.1389644943789974
  Donor2: 20 N 2.1770916512600014
  Palladacycle: 5-membered

[7/19] L16_new.xyz
  Overall: WARNING
  Donor1: 17 N 2.1064700144208084
  Donor2: 21 N 2.206978262473829
  Palladacycle: 6-membered

[8/19] L17_new.xyz
  Overall: WARNING
  Donor1: 17 N 2.0765219191065625
  Donor2: 21 N 

,Ligand,Total_Atoms,Ligand_Atom_Count,Donor1_Atom,Donor1_Element,Donor1_Pd_Distance_A,Donor2_Atom,Donor2_Element,Donor2_Pd_Distance_A,Donor_Selection_Check,...,BiteAngle_Check,SASA_Check,SolidAngle_Check,Sterimol_Check,Donor1_Pyr_Check,Donor2_Pyr_Check,XTB_Donor_Check,XTB_Check,NBO_Check,Overall_Check
0,L10_new,37,21,17,N,2.112539,20,N,2.127016,PASS,...,PASS,PASS,FAIL,PASS,FAIL,FAIL,FAIL,WARNING,NOT_CHECKED,WARNING
1,L11_new,41,25,20,N,2.136285,17,N,2.148657,PASS,...,PASS,PASS,FAIL,PASS,FAIL,FAIL,FAIL,WARNING,NOT_CHECKED,WARNING
2,L12_new,43,27,17,N,2.121125,20,N,2.285531,PASS,...,PASS,PASS,FAIL,PASS,FAIL,FAIL,FAIL,WARNING,NOT_CHECKED,WARNING
3,L13_new,43,27,17,N,2.108251,20,N,2.138895,PASS,...,PASS,PASS,FAIL,PASS,FAIL,FAIL,FAIL,WARNING,NOT_CHECKED,WARNING
4,L14-new,48,32,17,N,2.139124,20,N,2.164505,PASS,...,PASS,PASS,FAIL,PASS,FAIL,FAIL,FAIL,WARNING,NOT_CHECKED,WARNING
5,L15_new,38,22,17,N,2.138964,20,N,2.177092,PASS,...,PASS,PASS,FAIL,PASS,FAIL,FAIL,FAIL,WARNING,NOT_CHECKED,WARNING
6,L16_new,41,25,17,N,2.106470,21,N,2.206978,PASS,...,PASS,PASS,FAIL,PASS,FAIL,FAIL,FAIL,WARNING,NOT_CHECKED,WARNING
7,L17_new,43,27,17,N,2.076522,21,N,2.224884,PASS,...,PASS,PASS,FAIL,PASS,FAIL,FAIL,FAIL,WARNING,NOT_CHECKED,WARNING
8,L18_new,46,30,17,N,2.075743,21,N,2.219361,PASS,...,PASS,PASS,FAIL,PASS,FAIL,FAIL,FAIL,WARNING,NOT_CHECKED,WARNING
9,L19_new,43,27,17,N,2.076943,21,N,2.219686,PASS,...,PASS,PASS,FAIL,PASS,FAIL,FAIL,FAIL,WARNING,NOT_CHECKED,WARNING




FILES REQUIRING MANUAL INSPECTION

FILE: L10_new.xyz
STATUS: WARNING
WARNINGS: Sterimol_Warning: Pd-oriented/custom Sterimol; NOT standard H-dummy literature Sterimol. || Palladacycle_Warning: Ligand-only geometry gives path: 17-18-19-20 || XTB_Donor_Warning: Required executables not found in path: xtb || Donor1_Pyr_Warning: 2 neighbors. 3 expected. || Donor2_Pyr_Warning: 2 neighbors. 3 expected. || XTB_Warning: GFN2-xTB on ligand atoms 17->last. Coordinates are taken directly from the Pd TS; the isolated ligand was NOT geometry optimized. || NBO_Warning: NBO descriptors require Gaussian/NBO output. XYZ alone cannot provide E(2).

FILE: L11_new.xyz
STATUS: WARNING
WARNINGS: Sterimol_Warning: Pd-oriented/custom Sterimol; NOT standard H-dummy literature Sterimol. || Palladacycle_Warning: Ligand-only geometry gives path: 20-19-18-17 || XTB_Donor_Warning: Required executables not found in path: xtb || Donor1_Pyr_Warning: 2 neighbors. 3 expected. || Donor2_Pyr_Warning: 2 neighbors. 3 expe